In [ ]:
import requests
import os
import csv
from itertools import product
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

TICKERS = [
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSB","SSI","STB",
    "TCB","TPB","VCB","VHM","VIB","VIC","VJC","VNM","VPB","VRE",
    "VND","VIX","PVS","PVD","NLG","KDH","DXG","DIG","KBC","DCM",
    "DPM","REE","GEX","EIB","OCB","LPB","SCS","CTR","BSR","IDC"
]

TICKER_EXCHANGE = {
    "ACB":"HOSE","BCM":"HOSE","BID":"HOSE","BVH":"HOSE","CTG":"HOSE",
    "FPT":"HOSE","GAS":"HOSE","GVR":"HOSE","HDB":"HOSE","HPG":"HOSE",
    "MBB":"HOSE","MSN":"HOSE","MWG":"HOSE","PLX":"HOSE","POW":"HOSE",
    "SAB":"HOSE","SSI":"HOSE","STB":"HOSE","TCB":"HOSE","TPB":"HOSE",
    "VCB":"HOSE","VHM":"HOSE","VIB":"HOSE","VIC":"HOSE","VJC":"HOSE",
    "VNM":"HOSE","VPB":"HOSE","VRE":"HOSE","REE":"HOSE","EIB":"HOSE",
    "NLG":"HOSE","KDH":"HOSE","DXG":"HOSE","DIG":"HOSE","DCM":"HOSE",
    "DPM":"HOSE","GEX":"HOSE","CTR":"HOSE","BSR":"HOSE",
    "SHB":"HNX","SSB":"HNX","PVS":"HNX","KBC":"HNX","SCS":"HNX",
    "IDC":"HNX","OCB":"HNX",
    "VND":"UPCOM","VIX":"UPCOM","PVD":"UPCOM","LPB":"UPCOM",
}

# ↓↓↓ 3 THAY ĐỔI CHÍNH ↓↓↓
YEARS       = [2025]                          # ← Chỉ năm 2025
QUARTERS    = [1, 2, 3, 4]                   # ← Đủ 4 quý
QUARTER_MAP = {1:"QUY%201", 2:"QUY%202",
               3:"QUY%203", 4:"QUY%204"}

# Period tương ứng từng quý (chặt chẽ hơn, bỏ 6T/9T không cần)
QUARTER_PERIOD_MAP = {
    1: ["Q1"],           # Quý 1
    2: ["Q2", "6T"],     # Quý 2 có thể dùng 6T
    3: ["Q3", "9T"],     # Quý 3 có thể dùng 9T
    4: ["Q4"],           # Quý 4
}

TYPES      = ["Soatxet", "KiemToan"]
SCOPES     = ["Congtyme", "Hopnhat"]

HEADERS    = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
OUTPUT_DIR = "/content/drive/MyDrive/LandingAI/bctc_2025"        # ← Thư mục riêng cho 2025
MAX_WORKERS = 30                # ← Tăng lên vì ít URL hơn
DOWNLOAD    = True

# ─────────────────────────────────────────────
# BUILD URL
# ─────────────────────────────────────────────
def build_bctc_url(ticker, exchange, year, quarter, period, type_, scope):
    return (
        f"https://static2.vietstock.vn/data/"
        f"{exchange}/{year}/BCTC/VN/"
        f"{QUARTER_MAP[quarter]}/"
        f"{ticker}_Baocaotaichinh_{period}_{year}_{type_}_{scope}.pdf"
    )

def build_bctn_url(ticker, exchange, year):
    return (
        f"https://static2.vietstock.vn/data/"
        f"{exchange}/{year}/BCTN/VN/"
        f"{ticker}_Baocaothuongnien_{year}.pdf"
    )

# ─────────────────────────────────────────────
# CHECK URL
# ─────────────────────────────────────────────
def check_url(url: str) -> bool:
    try:
        resp = requests.head(url, timeout=6, headers=HEADERS)
        return resp.status_code == 200
    except Exception:
        return False

# ─────────────────────────────────────────────
# DOWNLOAD FILE
# ─────────────────────────────────────────────
def download_pdf(url: str, save_path: str) -> bool:
    try:
        if os.path.exists(save_path):
            return True
        resp = requests.get(url, timeout=30, headers=HEADERS)
        if resp.status_code == 200:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            with open(save_path, "wb") as f:
                f.write(resp.content)
            return True
    except Exception:
        pass
    return False

# ─────────────────────────────────────────────
# TẠO URL CHO 1 MÃ (chỉ 2025, 4 quý)
# ─────────────────────────────────────────────
def generate_all_urls(ticker: str, exchange: str) -> list[dict]:
    urls = []
    year = 2025

    for quarter in QUARTERS:
        periods = QUARTER_PERIOD_MAP[quarter]
        for period, type_, scope in product(periods, TYPES, SCOPES):
            url = build_bctc_url(ticker, exchange, year,
                                  quarter, period, type_, scope)
            urls.append({
                "ticker":    ticker,
                "exchange":  exchange,
                "year":      year,
                "quarter":   quarter,
                "period":    period,
                "type":      type_,
                "scope":     scope,
                "report":    "BCTC",
                "url":       url,
                "save_path": os.path.join(
                    OUTPUT_DIR, ticker,
                    f"{ticker}_{year}_Q{quarter}_{period}_{type_}_{scope}.pdf"
                ),
            })

    # BCTN 2025
    bctn_url = build_bctn_url(ticker, exchange, year)
    urls.append({
        "ticker":    ticker,
        "exchange":  exchange,
        "year":      year,
        "quarter":   None,
        "period":    "Annual",
        "type":      "BCTN",
        "scope":     "N/A",
        "report":    "BCTN",
        "url":       bctn_url,
        "save_path": os.path.join(
            OUTPUT_DIR, ticker,
            f"{ticker}_{year}_Thuongnien.pdf"
        ),
    })
    return urls

# ─────────────────────────────────────────────
# WORKER
# ─────────────────────────────────────────────
def process_one(item: dict) -> dict | None:
    if not check_url(item["url"]):
        return None
    item["found"] = True
    if DOWNLOAD:
        item["downloaded"] = download_pdf(item["url"], item["save_path"])
    return item

# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def run_all():
    start_time = datetime.now()

    # Tính trước số URL
    all_items = []
    for ticker in TICKERS:
        exchange = TICKER_EXCHANGE.get(ticker, "HOSE")
        all_items.extend(generate_all_urls(ticker, exchange))

    total_check = len(all_items)

    print("=" * 65)
    print(f"🕐 Bắt đầu    : {start_time.strftime('%d/%m/%Y %H:%M:%S')}")
    print(f"📋 Tổng mã    : {len(TICKERS)} mã CK")
    print(f"📅 Năm        : 2025 | 4 quý (Q1→Q4)")
    print(f"🔍 Tổng URL   : {total_check:,}")
    print(f"⚡ Thread     : {MAX_WORKERS}")
    print(f"💾 Tải file   : {'CÓ' if DOWNLOAD else 'KHÔNG'}")
    print(f"📁 Thư mục    : ./{OUTPUT_DIR}/")
    print("=" * 65 + "\n")

    found_all  = []
    done_count = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_one, item): item
                   for item in all_items}

        for future in as_completed(futures):
            done_count += 1
            result = future.result()
            if result:
                found_all.append(result)
                dl_icon  = "💾" if result.get("downloaded") else "🔗"
                q_label  = f"Q{result['quarter']}" if result['quarter'] else "Annual"
                print(
                    f"  {dl_icon} {result['ticker']:5s} | "
                    f"{q_label:6s} | "
                    f"{result['period']:3s} | "
                    f"{result['type']:9s} | "
                    f"{result['scope']}"
                )

            if done_count % 200 == 0:
                pct = done_count / total_check * 100
                print(f"\n  📊 {done_count:,}/{total_check:,} "
                      f"({pct:.0f}%) | ✅ {len(found_all)} file\n")

    elapsed = (datetime.now() - start_time).seconds

    # ── Tổng kết ──────────────────────────────
    print(f"\n{'='*65}")
    print(f"✅ HOÀN THÀNH!")
    print(f"📊 Tìm được   : {len(found_all)} file PDF")
    print(f"⏱️  Thời gian  : {elapsed}s")
    if DOWNLOAD:
        dl_ok = sum(1 for r in found_all if r.get("downloaded"))
        print(f"💾 Đã tải     : {dl_ok}/{len(found_all)} file")
    print(f"{'='*65}\n")

    # ── Thống kê theo mã ──────────────────────
    print("📋 THỐNG KÊ THEO MÃ:\n")
    ticker_stats = {}
    for r in found_all:
        t = r["ticker"]
        ticker_stats.setdefault(t, {
            "Q1":0,"Q2":0,"Q3":0,"Q4":0,"bctn":0,"total":0
        })
        if r["report"] == "BCTN":
            ticker_stats[t]["bctn"] += 1
        else:
            key = f"Q{r['quarter']}"
            ticker_stats[t][key] = ticker_stats[t].get(key, 0) + 1
        ticker_stats[t]["total"] += 1

    print(f"  {'Mã':6s} | {'Q1':>4} | {'Q2':>4} | {'Q3':>4} | {'Q4':>4} | {'BCTN':>4} | {'Tổng':>5}")
    print(f"  {'─'*55}")
    for ticker in TICKERS:
        s = ticker_stats.get(ticker, {})
        if not s:
            print(f"  {ticker:6s} | ❌ Không tìm thấy")
        else:
            print(
                f"  {ticker:6s} | "
                f"{s.get('Q1',0):>4} | "
                f"{s.get('Q2',0):>4} | "
                f"{s.get('Q3',0):>4} | "
                f"{s.get('Q4',0):>4} | "
                f"{s.get('bctn',0):>4} | "
                f"{s.get('total',0):>5}"
            )

    # ── Lưu CSV ───────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, "found_urls_2025.csv")
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "ticker","exchange","year","quarter",
            "period","type","scope","report","url","save_path"
        ])
        writer.writeheader()
        for r in found_all:
            writer.writerow({k: r.get(k,"") for k in writer.fieldnames})

    print(f"\n📄 CSV: {csv_path}")
    return found_all


found = run_all()

🕐 Bắt đầu    : 13/04/2026 15:51:24
📋 Tổng mã    : 50 mã CK
📅 Năm        : 2025 | 4 quý (Q1→Q4)
🔍 Tổng URL   : 1,250
⚡ Thread     : 30
💾 Tải file   : CÓ
📁 Thư mục    : .//content/drive/MyDrive/LandingAI/bctc_2025/

  💾 ACB   | Q2     | 6T  | Soatxet   | Congtyme
  💾 ACB   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 BID   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 BVH   | Q1     | Q1  | Soatxet   | Congtyme
  💾 BVH   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 BVH   | Q3     | 9T  | Soatxet   | Congtyme
  💾 BVH   | Q2     | 6T  | Soatxet   | Congtyme
  💾 BVH   | Q1     | Q1  | Soatxet   | Hopnhat
  💾 CTG   | Q2     | 6T  | Soatxet   | Congtyme
  💾 FPT   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 CTG   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 FPT   | Annual | Annual | BCTN      | N/A
  💾 GAS   | Q2     | 6T  | Soatxet   | Congtyme
  💾 BCM   | Q2     | 6T  | Soatxet   | Congtyme
  💾 GAS   | Q2     | 6T  | Soatxet   | Hopnhat
  💾 GAS   | Annual | Annual | BCTN      | N/A
  💾 GVR   | Q2     | 6T  | So

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
